## Imports

In [1]:
import numpy as np
import pandas as pd
import requests
import time

pd.set_option("display.max_rows", 200) # Set the maximum number of rows to display
pd.set_option("display.width", 120) # Set the width of the display


## Étape 1 — Liste des maladies GBD niveau 3

In [2]:
# Liste curatée de maladies GBD niveau 3, par catégorie parente (niveau 2) ---
# Sources : hiérarchie des causes GBD 2019/2021 (structure publique, cf. publications GBD /
# documentation IHME). Liste non exhaustive.

raw_data = {
    "Neurological disorders": [
        "Alzheimer's disease and other dementias",
        "Parkinson's disease",
        "Epilepsy",
        "Multiple sclerosis",
        "Motor neuron disease",
        "Migraine",
        "Tension-type headache",
        "Guillain-Barré syndrome",
        "Encephalitis",
        "Idiopathic developmental intellectual disability",
        "Headache disorders",
        "Tension-type headache",
    ],
    "Cardiovascular diseases": [
        "Ischemic heart disease",
        "Ischemic stroke",
        "Intracerebral hemorrhage",
        "Subarachnoid hemorrhage",
        "Hypertensive heart disease",
        "Cardiomyopathy and myocarditis",
        "Atrial fibrillation and flutter",
        "Rheumatic heart disease",
        "Aortic aneurysm",
        "Peripheral artery disease",
        "Endocarditis",
        "Non-rheumatic valvular heart disease",
    ],
    "Neoplasms": [
        "Breast cancer",
        "Prostate cancer",
        "Colon and rectum cancer",
        "Tracheal, bronchus, and lung cancer",
        "Ovarian cancer",
        "Cervical cancer",
        "Uterine cancer",
        "Liver cancer",
        "Stomach cancer",
        "Esophageal cancer",
        "Pancreatic cancer",
        "Non-Hodgkin lymphoma",
        "Leukemia",
        "Brain cancer",
        "Bladder cancer",
        "Kidney cancer",
        "Thyroid cancer",
        "Malignant skin melanoma",
        "Larynx cancer",
        "Multiple myeloma",
    ],
    "Endocrine, metabolic, blood and immune disorders": [
        "Diabetes mellitus type 1",
        "Diabetes mellitus type 2",
        "Gout",
        "Thyroid disorders",
        "Systemic lupus erythematosus",
        "Sickle cell disorders",
        "Thalassemias",
        "G6PD deficiency",
        "Addison's disease",
        "Other endocrine, metabolic, blood, and immune disorders",
    ],
    "Musculoskeletal disorders": [
        "Rheumatoid arthritis",
        "Osteoarthritis",
        "Low back pain",
        "Neck pain",
        "Gout (musculoskeletal manifestations)",
        "Juvenile idiopathic arthritis",
        "Fibromyalgia",
        "Other musculoskeletal disorders",
    ],
    "Mental disorders": [
        "Depressive disorders",
        "Anxiety disorders",
        "Bipolar disorder",
        "Schizophrenia",
        "Eating disorders",
        "Autism spectrum disorders",
        "Attention-deficit/hyperactivity disorder",
        "Conduct disorder",
        "Post-traumatic stress disorder",
        "Obsessive-compulsive disorder",
    ],
    "Chronic respiratory diseases": [
        "Chronic obstructive pulmonary disease",
        "Asthma",
        "Interstitial lung disease",
        "Pulmonary sarcoidosis"
        
    ],
    "Digestive diseases": [
        "Cirrhosis and other chronic liver diseases",
        "Gastritis and duodenitis",
        "Peptic ulcer disease",
        "Inflammatory bowel disease",
        "Pancreatitis",
        "Gallbladder and biliary diseases",
        "Appendicitis",
        "Vascular intestinal disorders",
        "Irritable bowel syndrome",
    ],
    "Skin and subcutaneous diseases": [
        "Dermatitis",
        "Psoriasis",
        "Acne vulgaris",
        "Alopecia areata",
        "Urticaria",
        "Scabies",
        "Fungal skin diseases",
        "Decubitus ulcer",
        "Other skin and subcutaneous diseases",
    ],
    "Sense organ diseases": [
        "Age-related and other hearing loss",
        "Cataract",
        "Glaucoma",
        "Age-related macular degeneration",
        "Refraction disorders",
        "Other vision loss",
    ],
    "Urinary, gynecological and reproductive diseases": [
        "Urinary tract infections",
        "Urolithiasis",
        "Benign prostatic hyperplasia",
        "Male infertility",
        "Interstitial nephritis",
        "Endometriosis",
        "Uterine fibroids",
        "Genital prolapse",
        "Premenstrual syndrome",
        "Female infertility",
        "Polycystic ovary syndrome",
    ],
}

rows = []
for category, diseases_list in raw_data.items():
    for name in diseases_list:
        rows.append({
            "cause_name": name,
            "parent": category,
        })

diseases = pd.DataFrame(rows)
diseases

,cause_name,parent
0,Alzheimer's disease and other dementias,Neurological disorders
1,Parkinson's disease,Neurological disorders
2,Epilepsy,Neurological disorders
3,Multiple sclerosis,Neurological disorders
4,Motor neuron disease,Neurological disorders
5,Migraine,Neurological disorders
6,Tension-type headache,Neurological disorders
7,Guillain-Barré syndrome,Neurological disorders
8,Encephalitis,Neurological disorders
9,Idiopathic developmental intellectual disability,Neurological disorders


## Étape 2 — Enrichissement automatique : nombre de publications scientifiques

Parmi les variables listées plus haut, le **nombre de publications** est la seule qu'on peut
récupérer entièrement automatiquement, via deux API gratuites et sans clé :

**PubMed / NCBI E-utilities** : `esearch.fcgi` renvoie un simple compte de résultats pour une
  requête donnée.

Les autres variables (proportion de femmes atteintes, financement, délai diagnostique) ne sont
**pas** automatisables de façon fiable — elles nécessitent une recherche ciblée par maladie. On
crée donc des colonnes vides pour elles, à remplir manuellement grâce à une recherche via perplexity sur des sites tels que ameli, gbd, WHO, ...

In [4]:
def count_pubmed(query: str, retries: int = 2, timeout: int = 10) -> float:
    """Retourne le nombre de résultats PubMed pour une requête, ou NaN en cas d'échec."""
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    params = {"db": "pubmed", "term": query, "retmode": "json"}
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return int(r.json()["esearchresult"]["count"])
        except Exception:
            time.sleep(1)
    return np.nan

pubmed_counts = []
for disease in diseases["cause_name"]:
    pubmed_counts.append(count_pubmed(disease))
    time.sleep(0.34)  # reste sous la limite de ~3 requêtes/seconde sans clé API NCBI

diseases["publications_pubmed"] = pubmed_counts

n_echecs = sum(1 for count in pubmed_counts if pd.isna(count))
if n_echecs == len(diseases):
    print("Toutes les requêtes ont échoué : le réseau semble bloqué dans cet environnement. ")
else:
    print(f"Publications récupérées pour {len(diseases) - n_echecs}/{len(diseases)} maladies.")

pubmed_df = diseases[["cause_name", "parent", "publications_pubmed"]] \
    .sort_values("publications_pubmed", ascending=False)

pubmed_df

Publications récupérées pour 111/111 maladies.


,cause_name,parent,publications_pubmed
12,Ischemic heart disease,Cardiovascular diseases,618602
24,Breast cancer,Neoplasms,587604
80,Pancreatitis,Digestive diseases,466772
36,Leukemia,Neoplasms,403365
31,Liver cancer,Neoplasms,377782
37,Brain cancer,Neoplasms,293028
73,Asthma,Chronic respiratory diseases,244204
25,Prostate cancer,Neoplasms,241448
45,Diabetes mellitus type 2,"Endocrine, metabolic, blood and immune disorders",233523
2,Epilepsy,Neurological disorders,205751
